<a href="https://www.kaggle.com/code/alinaliaquat/fire-detection?scriptVersionId=344293706" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

### Install Dependencies

In [ ]:
!pip install ultralytics roboflow opencv-python matplotlib pandas seaborn

### Import Libraries

In [ ]:
import torch
import cv2
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from ultralytics import YOLO
from pathlib import Path
import pandas as pd
import seaborn as sns
from IPython.display import display, Video
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

### Dataset Validation

In [ ]:
import os
import yaml

dataset_path = "/kaggle/input/datasets/alinaliaquat/fire-smoke"

train_img_path = os.path.join(dataset_path, "train/images")
train_lbl_path = os.path.join(dataset_path, "train/labels")
val_img_path = os.path.join(dataset_path, "valid/images")
val_lbl_path = os.path.join(dataset_path, "valid/labels")
test_img_path = os.path.join(dataset_path, "test/images")
test_lbl_path = os.path.join(dataset_path, "test/labels")

print(f"Training images: {len(os.listdir(train_img_path))}")
print(f"Training labels: {len(os.listdir(train_lbl_path))}")
print(f"Validation images: {len(os.listdir(val_img_path))}")
print(f"Validation labels: {len(os.listdir(val_lbl_path))}")
print(f"Test images: {len(os.listdir(test_img_path))}")
print(f"Test labels: {len(os.listdir(test_lbl_path))}")

sample_img = os.listdir(train_img_path)[0]
sample_img_path = os.path.join(train_img_path, sample_img)
img = cv2.imread(sample_img_path)
print(f"Sample image shape: {img.shape}")
print(f"Class names: ['Fire', 'Smoke']")

### Create Data Yaml

In [ ]:
data_yaml = {
    'train': '/kaggle/input/datasets/alinaliaquat/fire-smoke/train/images',
    'val': '/kaggle/input/datasets/alinaliaquat/fire-smoke/valid/images',
    'test': '/kaggle/input/datasets/alinaliaquat/fire-smoke/test/images',
    'nc': 2,
    'names': ['Fire', 'Smoke']
}

with open('/kaggle/working/data.yaml', 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print("data.yaml created successfully!")
print("\nContent:")
print(yaml.dump(data_yaml, default_flow_style=False))

### Visualize Sample Annotations

In [ ]:
def visualize_annotations(img_path, label_path, class_names=['Fire', 'Smoke']):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(img)
    
    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            lines = f.readlines()
        
        for line in lines:
            parts = line.strip().split()
            if len(parts) >= 5:
                cls_id = int(parts[0])
                x_center = float(parts[1]) * w
                y_center = float(parts[2]) * h
                box_w = float(parts[3]) * w
                box_h = float(parts[4]) * h
                
                x = x_center - box_w/2
                y = y_center - box_h/2
                
                color = 'red' if cls_id == 0 else 'orange'
                rect = Rectangle((x, y), box_w, box_h, 
                               linewidth=3, edgecolor=color, facecolor='none')
                ax.add_patch(rect)
                ax.text(x, y-5, class_names[cls_id], color=color, 
                       fontsize=12, weight='bold')
    
    ax.set_title(f"Sample Image with Annotations")
    ax.axis('off')
    plt.tight_layout()
    plt.show()

sample_img = os.listdir(train_img_path)[2]
sample_label = sample_img.replace('.jpg', '.txt').replace('.jpeg', '.txt').replace('.png', '.txt')
label_path = os.path.join(train_lbl_path, sample_label)

if os.path.exists(os.path.join(train_lbl_path, sample_label)):
    visualize_annotations(os.path.join(train_img_path, sample_img), 
                         os.path.join(train_lbl_path, sample_label))
else:
    print(f"Label file not found for {sample_img}")

### Training Configuration

In [ ]:
MODEL_SIZE = 'n'  # 'n' for nano, 's' for small, 'm' for medium
EPOCHS = 50
BATCH_SIZE = 16
IMG_SIZE = 640
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
WORKERS = 4
PATIENCE = 20
PROJECT_NAME = 'fire_smoke_detector'
EXPERIMENT_NAME = 'experiment_v1'

print(f"Configuration:")
print(f"Model Size: YOLOv8{MODEL_SIZE}")
print(f"Epochs: {EPOCHS}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Image Size: {IMG_SIZE}")
print(f"Device: {DEVICE}")
print(f"Patience: {PATIENCE}")

### Initialize and Train Model

In [ ]:
model = YOLO(f'yolov8{MODEL_SIZE}.pt')

results = model.train(
    data='/kaggle/working/data.yaml',
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMG_SIZE,
    device=DEVICE,
    workers=WORKERS,
    patience=PATIENCE,
    project=PROJECT_NAME,
    name=EXPERIMENT_NAME,
    exist_ok=True,
    pretrained=True,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,
    box=7.5,
    cls=0.5,
    dfl=1.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=0.0,
    translate=0.1,
    scale=0.5,
    shear=0.0,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.0,
    copy_paste=0.0
)

print("Training completed successfully!")

### Evaluate Model Performance

In [ ]:
from ultralytics import YOLO
import torch

# Correct path - model runs/detect folder mein save hua hai
best_model_path = f'/kaggle/working/runs/detect/{PROJECT_NAME}/{EXPERIMENT_NAME}/weights/best.pt'

# Load model
model = YOLO(best_model_path)
print(f"Model loaded from: {best_model_path}")

# Validate
metrics = model.val(data='/kaggle/working/data.yaml', device=DEVICE)

print("\n" + "="*50)
print("EVALUATION METRICS")
print("="*50)
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision (P): {metrics.box.mp:.4f}")
print(f"Recall (R): {metrics.box.mr:.4f}")
if metrics.box.mp > 0 and metrics.box.mr > 0:
    f1 = 2 * (metrics.box.mp * metrics.box.mr) / (metrics.box.mp + metrics.box.mr + 1e-6)
    print(f"F1 Score: {f1:.4f}")
print("="*50)

# Class-wise metrics using maps attribute (mAP@0.5:0.95)
print("\nClass-wise Performance (mAP@0.5:0.95):")
class_names = ['Fire', 'Smoke']
if hasattr(metrics.box, 'maps'):
    for i, class_name in enumerate(class_names):
        if i < len(metrics.box.maps):
            print(f"{class_name}: {metrics.box.maps[i]:.4f}")

# Get mAP@0.5 for each class using ap50
print("\nClass-wise Performance (mAP@0.5):")
if hasattr(metrics.box, 'ap50'):
    if isinstance(metrics.box.ap50, (list, tuple)) and len(metrics.box.ap50) >= 2:
        for i, class_name in enumerate(class_names):
            if i < len(metrics.box.ap50):
                print(f"{class_name}: {metrics.box.ap50[i]:.4f}")
    elif hasattr(metrics.box.ap50, '__len__') and len(metrics.box.ap50) >= 2:
        for i, class_name in enumerate(class_names):
            if i < len(metrics.box.ap50):
                print(f"{class_name}: {metrics.box.ap50[i]:.4f}")
    else:
        print("ap50 data format not accessible directly")
else:
    print("ap50 attribute not found")
print("="*50)

### Plot Training Metrics

In [ ]:
def plot_training_metrics():
    results_file = f'/kaggle/working/runs/detect/{PROJECT_NAME}/{EXPERIMENT_NAME}/results.csv'
    
    if os.path.exists(results_file):
        df = pd.read_csv(results_file)
        
        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        
        metrics_to_plot = [
            ('train/box_loss', 'Box Loss', 'Training Box Loss'),
            ('train/cls_loss', 'Class Loss', 'Training Class Loss'),
            ('train/dfl_loss', 'DFL Loss', 'Training DFL Loss'),
            ('metrics/precision(B)', 'Precision', 'Precision'),
            ('metrics/recall(B)', 'Recall', 'Recall'),
            ('metrics/mAP50(B)', 'mAP@0.5', 'mAP@0.5')
        ]
        
        axes = axes.flatten()
        
        for idx, (metric, ylabel, title) in enumerate(metrics_to_plot):
            if metric in df.columns:
                axes[idx].plot(df['epoch'], df[metric], linewidth=2, color='blue')
                axes[idx].set_xlabel('Epoch', fontsize=12)
                axes[idx].set_ylabel(ylabel, fontsize=12)
                axes[idx].set_title(title, fontsize=14, weight='bold')
                axes[idx].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        print("\nBest Values:")
        print(f"Best mAP@0.5: {df['metrics/mAP50(B)'].max():.4f} at epoch {df['metrics/mAP50(B)'].idxmax()}")
        print(f"Best Precision: {df['metrics/precision(B)'].max():.4f} at epoch {df['metrics/precision(B)'].idxmax()}")
        print(f"Best Recall: {df['metrics/recall(B)'].max():.4f} at epoch {df['metrics/recall(B)'].idxmax()}")
    else:
        print("Results file not found!")

plot_training_metrics()

### Confusion Matrix & F1 Curve

In [ ]:
from IPython.display import Image
import os

confusion_path = f'/kaggle/working/runs/detect/{PROJECT_NAME}/{EXPERIMENT_NAME}/confusion_matrix.png'
f1_path = f'/kaggle/working/runs/detect/{PROJECT_NAME}/{EXPERIMENT_NAME}/F1_curve.png'
results_path = f'/kaggle/working/runs/detect/{PROJECT_NAME}/{EXPERIMENT_NAME}'

# Check for all available plots
plot_files = ['confusion_matrix.png', 'F1_curve.png', 'P_curve.png', 'R_curve.png', 'PR_curve.png', 'labels.jpg']

print("Available plots:")
for plot in plot_files:
    full_path = os.path.join(results_path, plot)
    if os.path.exists(full_path):
        print(f"✓ {plot}")
    else:
        print(f"✗ {plot}")

print("\nDisplaying available plots:")
if os.path.exists(confusion_path):
    print("\nConfusion Matrix:")
    display(Image(filename=confusion_path))
else:
    print("\nConfusion matrix not found")

if os.path.exists(f1_path):
    print("\nF1 Curve:")
    display(Image(filename=f1_path))
else:
    print("F1 curve not found")

# Display Precision-Recall curve if available
pr_curve_path = os.path.join(results_path, 'PR_curve.png')
if os.path.exists(pr_curve_path):
    print("\nPrecision-Recall Curve:")
    display(Image(filename=pr_curve_path))

### Save Model for Future Use

In [ ]:
import shutil
import os

# Source model path (where training saved it)
source_path = '/kaggle/working/runs/detect/fire_smoke_detector/experiment_v1/weights/best.pt'

# Destination path (for easy access)
destination_path = '/kaggle/working/fire_smoke_best_model.pt'

# Copy the model
if os.path.exists(source_path):
    shutil.copy(source_path, destination_path)
    print(f"✓ Model copied successfully!")
    print(f"  Source: {source_path}")
    print(f"  Destination: {destination_path}")
    print(f"  Size: {os.path.getsize(destination_path) / 1024 / 1024:.2f} MB")
else:
    print(f"✗ Model not found at: {source_path}")

### Load Model for Inference (NO RETRAINING)

In [ ]:
from ultralytics import YOLO
import torch
import os

def load_model():
    # Try multiple paths
    paths = [
        '/kaggle/input/datasets/alinaliaquat/smoke-model/smoke.pt'
    ]
    
    for path in paths:
        if os.path.exists(path):
            model = YOLO(path)
            print(f"✓ Model loaded from: {path}")
            print(f"✓ Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
            return model
    
    print("✗ No model found! Please train first.")
    return None

# Load the model
MODEL = load_model()

### Inference on Single Image

In [ ]:
def predict_single_image(model, image_path, conf_threshold=0.25):
    if not os.path.exists(image_path):
        print(f"Image not found: {image_path}")
        return None
    
    results = model(image_path, conf=conf_threshold, device=DEVICE)
    
    if len(results) > 0 and results[0].boxes is not None:
        img = cv2.imread(image_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        fig, ax = plt.subplots(figsize=(12, 12))
        ax.imshow(img)
        
        boxes = results[0].boxes
        for box in boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            conf = box.conf[0].cpu().numpy()
            cls_id = int(box.cls[0].cpu().numpy())
            class_name = 'Fire' if cls_id == 0 else 'Smoke'
            color = 'red' if cls_id == 0 else 'orange'
            
            rect = Rectangle((x1, y1), x2-x1, y2-y1, 
                           linewidth=3, edgecolor=color, facecolor='none')
            ax.add_patch(rect)
            
            label = f"{class_name}: {conf:.2f}"
            ax.text(x1, y1-10, label, color=color, fontsize=14, weight='bold',
                   bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))
        
        ax.set_title(f"Detection Results\nTotal Detections: {len(boxes)}", fontsize=16, weight='bold')
        ax.axis('off')
        plt.tight_layout()
        plt.show()
        
        return results
    else:
        print("No detections found")
        return None

# Test on a sample image
test_img = os.listdir(test_img_path)[0]
predict_single_image(MODEL, os.path.join(test_img_path, test_img))

### Test on Video (Configure Path Here Only)

In [ ]:
import cv2
import os
import numpy as np
from ultralytics import YOLO
from IPython.display import Video, display
import torch

def predict_video_premium(model, video_path, output_path=None, conf_threshold=0.25):
    """
    Process video with premium transparent bounding boxes and save output
    """
    if not os.path.exists(video_path):
        print(f"Video not found: {video_path}")
        return None
    
    if model is None:
        print("Model not loaded!")
        return None
    
    # Save to working directory
    if output_path is None:
        base_name = os.path.basename(video_path)
        name_without_ext = os.path.splitext(base_name)[0]
        output_path = f"/kaggle/working/{name_without_ext}_output.mp4"
    
    # Open video
    cap = cv2.VideoCapture(video_path)
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    # Video writer
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
    # Premium colors (BGR format) - Darker colors with white text
    COLORS = {
        'Fire': {
            'box': (0, 0, 200),        # Dark Red (more visible)
            'text': (255, 255, 255),   # White text (visible on dark bg)
            'bg': (0, 0, 0)            # Black background for label
        },
        'Smoke': {
            'box': (0, 140, 255),      # Dark Orange (more visible)
            'text': (255, 255, 255),   # White text (visible on dark bg)
            'bg': (0, 0, 0)            # Black background for label
        }
    }
    
    frame_count = 0
    total_detections = 0
    fire_detections = 0
    smoke_detections = 0
    
    print("="*60)
    print("FIRE & SMOKE DETECTION - VIDEO PROCESSING")
    print("="*60)
    print(f"Input:  {os.path.basename(video_path)}")
    print(f"Output: {os.path.basename(output_path)}")
    print(f"Frames: {total_frames}, FPS: {fps}, Size: {width}x{height}")
    print("-"*60)
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        # Run inference
        results = model(frame, conf=conf_threshold, device=0)
        
        # Create annotated frame with premium transparent boxes
        annotated = frame.copy()
        
        if results[0].boxes is not None:
            boxes = results[0].boxes
            
            for box in boxes:
                # Get coordinates
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
                conf = box.conf[0].cpu().numpy()
                cls_id = int(box.cls[0].cpu().numpy())
                class_name = 'Fire' if cls_id == 0 else 'Smoke'
                
                # Count detections
                total_detections += 1
                if cls_id == 0:
                    fire_detections += 1
                else:
                    smoke_detections += 1
                
                color = COLORS[class_name]['box']
                text_color = COLORS[class_name]['text']
                bg_color = COLORS[class_name]['bg']
                
                # ==========================================
                # CLEAN TRANSPARENT BOUNDING BOX - THICKER
                # ==========================================
                
                # 1. Outer glow (subtle)
                for i in range(3, 0, -1):
                    cv2.rectangle(
                        annotated,
                        (x1 - i, y1 - i),
                        (x2 + i, y2 + i),
                        color,
                        thickness=1,
                        lineType=cv2.LINE_AA
                    )
                
                # 2. Main thick box - THICKNESS INCREASED TO 4
                cv2.rectangle(
                    annotated,
                    (x1, y1),
                    (x2, y2),
                    color,
                    thickness=4,          # Increased from 2 to 4
                    lineType=cv2.LINE_AA
                )
                
                # 3. Corner accents (bigger for visibility)
                corner_len = min(20, (x2-x1)//5, (y2-y1)//5)
                # Top-left
                cv2.line(annotated, (x1, y1), (x1 + corner_len, y1), color, 3, cv2.LINE_AA)
                cv2.line(annotated, (x1, y1), (x1, y1 + corner_len), color, 3, cv2.LINE_AA)
                # Top-right
                cv2.line(annotated, (x2, y1), (x2 - corner_len, y1), color, 3, cv2.LINE_AA)
                cv2.line(annotated, (x2, y1), (x2, y1 + corner_len), color, 3, cv2.LINE_AA)
                # Bottom-left
                cv2.line(annotated, (x1, y2), (x1 + corner_len, y2), color, 3, cv2.LINE_AA)
                cv2.line(annotated, (x1, y2), (x1, y2 - corner_len), color, 3, cv2.LINE_AA)
                # Bottom-right
                cv2.line(annotated, (x2, y2), (x2 - corner_len, y2), color, 3, cv2.LINE_AA)
                cv2.line(annotated, (x2, y2), (x2, y2 - corner_len), color, 3, cv2.LINE_AA)
                
                # 4. Label with larger size and better visibility
                label = f"{class_name} {conf:.1%}"
                
                # BIGGER FONT SIZE - Increased from 0.55 to 0.8
                (label_w, label_h), baseline = cv2.getTextSize(
                    label,
                    cv2.FONT_HERSHEY_DUPLEX,
                    0.8,    # Increased font size
                    2       # Thicker text
                )
                
                # Position label above box
                label_y = max(y1 - 10, label_h + 15)
                label_x = x1
                
                # Ensure label stays within frame
                if label_y < label_h + 15:
                    label_y = y1 + label_h + 15
                
                # EXTRA PADDING FOR LABEL - Increased padding
                padding = 10
                label_bg_x1 = label_x - padding
                label_bg_y1 = label_y - label_h - padding
                label_bg_x2 = label_x + label_w + padding
                label_bg_y2 = label_y + padding
                
                # Semi-transparent background for label
                overlay = annotated.copy()
                cv2.rectangle(
                    overlay,
                    (label_bg_x1, label_bg_y1),
                    (label_bg_x2, label_bg_y2),
                    (0, 0, 0),
                    thickness=-1
                )
                cv2.addWeighted(overlay, 0.7, annotated, 0.3, 0, annotated)
                
                # Border around label (thicker)
                cv2.rectangle(
                    annotated,
                    (label_bg_x1, label_bg_y1),
                    (label_bg_x2, label_bg_y2),
                    color,
                    thickness=2,
                    lineType=cv2.LINE_AA
                )
                
                # Bigger dot indicator
                cv2.circle(
                    annotated,
                    (label_x + padding, label_y - label_h//2),
                    5,
                    color,
                    thickness=-1,
                    lineType=cv2.LINE_AA
                )
                
                # BIGGER TEXT - White color for visibility
                cv2.putText(
                    annotated,
                    label,
                    (label_x + padding + 12, label_y - 2),
                    cv2.FONT_HERSHEY_DUPLEX,
                    0.8,           # Bigger font
                    text_color,    # White text
                    2,             # Thicker text
                    cv2.LINE_AA
                )
        
        # Write frame
        out.write(annotated)
        frame_count += 1
        
        # Progress
        if frame_count % 50 == 0 or frame_count == total_frames:
            print(f"Progress: {frame_count}/{total_frames} frames | Detections: {total_detections}")
    
    # Release resources
    cap.release()
    out.release()
    
    print("-"*60)
    print("VIDEO PROCESSING COMPLETE")
    print("-"*60)
    print(f"Frames processed:  {frame_count}")
    print(f"Total detections:  {total_detections}")
    print(f"  - Fire:  {fire_detections}")
    print(f"  - Smoke: {smoke_detections}")
    print(f"Output saved to:   {output_path}")
    print("="*60)
    
    return output_path


# ============================================================
# VIDEO PATH
# ============================================================
VIDEO_PATH = "/kaggle/input/datasets/alinaliaquat/fire-videos/f1.mp4"


# ============================================================
# RUN
# ============================================================
if MODEL is not None:
    if os.path.exists(VIDEO_PATH):
        output = predict_video_premium(MODEL, VIDEO_PATH, conf_threshold=0.25)
        
        # Display the saved video
        if output and os.path.exists(output):
            print("\nDisplaying output video:")
            display(Video(output, width=800))
    else:
        print(f"Video not found at: {VIDEO_PATH}")
        print("\nAvailable video files:")
        video_dir = os.path.dirname(VIDEO_PATH)
        if os.path.exists(video_dir):
            for f in os.listdir(video_dir):
                if f.endswith(('.mp4', '.avi', '.mov', '.mkv')):
                    print(f"  - {f}")
else:
    print("Model not loaded. Please run Cell 12 first.")

### Batch Inference on Test Images

In [ ]:
def batch_predict(model, image_folder, conf_threshold=0.25, max_images=20):
    if not os.path.exists(image_folder):
        print(f"Folder not found: {image_folder}")
        return
    
    image_files = [f for f in os.listdir(image_folder) 
                   if f.endswith(('.jpg', '.jpeg', '.png'))][:max_images]
    
    print(f"Processing {len(image_files)} images...")
    
    fig, axes = plt.subplots(4, 5, figsize=(20, 16))
    axes = axes.flatten()
    
    for idx, img_file in enumerate(image_files):
        img_path = os.path.join(image_folder, img_file)
        results = model(img_path, conf=conf_threshold, device=DEVICE)
        
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        axes[idx].imshow(img)
        axes[idx].axis('off')
        
        if results[0].boxes is not None:
            axes[idx].set_title(f"Detections: {len(results[0].boxes)}", color='green')
        else:
            axes[idx].set_title("No detections", color='red')
    
    for idx in range(len(image_files), len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()

batch_predict(MODEL, test_img_path, max_images=20)

###  Export Model for Deployment

In [ ]:
def export_model(model, format='onnx'):
    output_path = f'/kaggle/working/fire_smoke_model.{format}'
    
    if format == 'onnx':
        model.export(format='onnx', imgsz=640, opset=12, simplify=True)
    elif format == 'tflite':
        model.export(format='tflite', imgsz=640, int8=False)
    elif format == 'torchscript':
        model.export(format='torchscript', imgsz=640)
    
    print(f"Model exported to: /kaggle/working/fire_smoke_model.{format}")
    return f'/kaggle/working/fire_smoke_model.{format}'

export_model(MODEL, format='onnx')

### Performance Summary & Results

In [ ]:
print("\n" + "="*70)
print("FIRE & SMOKE DETECTION - TRAINING COMPLETE")
print("="*70)
print(f"Model Path: /kaggle/working/fire_smoke_best_model.pt")
print(f"Dataset Size: 4000 images")
print(f"Classes: Fire, Smoke")
print("\nModel Performance:")
print(f"- mAP@0.5: {metrics.box.map50:.4f}")
print(f"- mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"- Precision: {metrics.box.mp:.4f}")
print(f"- Recall: {metrics.box.mr:.4f}")
print("="*70)
print("\nTo test on new videos, update VIDEO_PATH in Cell 14")
print("Model will load automatically without retraining")
print("="*70)